# Structured output smoke test

Checks whether the **currently configured** LLM (`LLM_PROVIDER`, `LLM_MODEL` from `.env`) supports LangChain `with_structured_output()`, which the metadata agent uses in synthesis (`Player._synthesize_structured`).

Run all cells top-to-bottom. A passing run prints a validated Pydantic object; failures show the exception and optional fallback methods.

In [ ]:
import sys
import os
import traceback
from typing import Optional

sys.path.insert(0, "..")

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

from src.config import (
    LLM_PROVIDER,
    create_llm,
    get_config_summary,
    get_model_name,
)
from src.standards import get_schema_for_standard

print(get_config_summary())
print(f"Resolved model: {get_model_name()}")

## Minimal schema (fast probe)

Small schema to reduce token use. The agent uses full metadata schemas from `src/standards.py` in production.

In [ ]:
class MinimalMetadata(BaseModel):
    """Tiny stand-in for production metadata schemas."""
    title: str = Field(description="Dataset title")
    description: str = Field(description="One-sentence dataset description")
    format: Optional[str] = Field(default=None, description="File or data format")


## Probe `with_structured_output`

Same pattern as `Player._synthesize_structured`: prompt → `llm.with_structured_output(schema)` → invoke.

In [ ]:
SAMPLE_TASK = "Extract metadata for a CSV of bird observations in the Netherlands."
SAMPLE_CONTEXT = """
Analyst notes:
- File: observations.csv, 12 columns, ~50k rows
- Columns include species, date, latitude, longitude
- Coverage: Netherlands, 2018–2022
""".strip()


def run_structured_probe(
    schema: type[BaseModel],
    method: str | None = None,
    temperature: float = 0.0,
) -> BaseModel:
    llm = create_llm(temperature=temperature)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You synthesize analyst notes into the required JSON schema. Use concrete values from the notes."),
        ("human", "Task: {task}\n\nAnalyst notes:\n{context}\n\nFill every required field."),
    ])
    kwargs = {} if method is None else {"method": method}
    structured_llm = llm.with_structured_output(schema, **kwargs)
    chain = prompt | structured_llm
    return chain.invoke({"task": SAMPLE_TASK, "context": SAMPLE_CONTEXT})


def try_methods(schema: type[BaseModel], label: str) -> None:
    # LangChain method names vary by provider; None = provider default
    candidates: list[str | None] = [None, "json_schema", "function_calling", "json_mode"]
    seen: set[str | None] = set()
    methods = []
    for m in candidates:
        if m not in seen:
            seen.add(m)
            methods.append(m)

    print(f"\n{'=' * 60}")
    print(f"Schema: {schema.__name__} ({label})")
    print(f"Provider: {LLM_PROVIDER}  Model: {get_model_name()}")
    print(f"{'=' * 60}")

    any_ok = False
    for method in methods:
        tag = method or "(default)"
        print(f"\n--- method={tag} ---")
        try:
            result = run_structured_probe(schema, method=method)
            print("OK — parsed instance:")
            print(result.model_dump_json(indent=2))
            any_ok = True
            print(f"\n=> Structured output works with method={tag}")
            break
        except Exception as e:
            print(f"FAILED: {type(e).__name__}: {e}")
            traceback.print_exc(limit=2)

    if not any_ok:
        print("\n=> No method succeeded. This model/provider may not support structured output for this schema.")

In [ ]:
try_methods(MinimalMetadata, "minimal")


## Plain completion (baseline)

If structured output fails, verify the model responds at all without schema binding.

In [ ]:
llm = create_llm(temperature=0.0)
baseline = llm.invoke(
    "Reply with exactly one JSON object: {""title"": ""Test"", ""description"": ""baseline""}"
)
print("Baseline response content:")
print(getattr(baseline, "content", baseline))

In [1]:
import os
import traceback
from typing import Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

load_dotenv("../.env")

MODEL = os.getenv("LLM_MODEL", "default-text-large")
BASE_URL = os.getenv("SURF_API_BASE")
API_KEY = os.getenv("SURF_API_KEY")

if not BASE_URL or not API_KEY:
    raise ValueError("Set SURF_API_BASE and SURF_API_KEY in .env")


class DatasetInfo(BaseModel):
    title: str = Field(description="Short dataset title")
    description: str = Field(description="One-sentence description")


llm = ChatOpenAI(
    model=MODEL,
    temperature=0,
    openai_api_key=API_KEY,
    openai_api_base=BASE_URL,
)

prompt = (
    "From these notes, fill the schema fields.\n"
    "Notes: observations.csv, bird species, Netherlands, 2018-2022."
)

for method in (None, "function_calling", "json_schema", "json_mode"):
    tag = method or "(default)"
    print(f"\n--- {tag} ---")
    try:
        kwargs = {} if method is None else {"method": method}
        structured = llm.with_structured_output(DatasetInfo, **kwargs)
        out = structured.invoke(prompt)
        print("OK:", out.model_dump())
        break
    except Exception as e:
        print(f"FAILED: {type(e).__name__}: {e}")
        traceback.print_exc(limit=1)
else:
    print("\nStructured output failed for all methods.")
    print("Baseline (no schema):")
    print(llm.invoke("Reply with JSON: {\"title\": \"Test\", \"description\": \"ok\"}").content)

Python-dotenv could not parse statement starting at line 5



--- (default) ---
FAILED: ValueError: Structured Output response does not have a 'parsed' field nor a 'refusal' field. Received message:

content='' additional_kwargs={'parsed': None, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 41, 'total_tokens': 166, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Sehyo/Qwen3.5-122B-A10B-NVFP4', 'system_fingerprint': None, 'id': 'chatcmpl-trace_19f928ba-9b14-4538-9653-2a9466a10323', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e49d8-6c84-76b3-9c96-fe9d6543382b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 41, 'output_tokens': 125, 'total_tokens': 166, 'input_token_details': {}, 'output_token_details': {}}

--- function_calling ---


Traceback (most recent call last):
  File "/tmp/ipykernel_23035/4056871112.py", line 42, in <module>
    out = structured.invoke(prompt)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: Structured Output response does not have a 'parsed' field nor a 'refusal' field. Received message:

content='' additional_kwargs={'parsed': None, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 41, 'total_tokens': 166, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Sehyo/Qwen3.5-122B-A10B-NVFP4', 'system_fingerprint': None, 'id': 'chatcmpl-trace_19f928ba-9b14-4538-9653-2a9466a10323', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e49d8-6c84-76b3-9c96-fe9d6543382b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 41, 'output_tokens': 125, 'total_tokens': 166, 'input_token_details': {}, 'output_token_details': {}}


FAILED: BadRequestError: Error code: 400 - {'message': '[\'{"error":{"message":"","type":"InternalServerError","param":null,"code":500}}\']', 'error': 'Bad Request'}

--- json_schema ---


Traceback (most recent call last):
  File "/tmp/ipykernel_23035/4056871112.py", line 42, in <module>
    out = structured.invoke(prompt)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
openai.BadRequestError: Error code: 400 - {'message': '[\'{"error":{"message":"","type":"InternalServerError","param":null,"code":500}}\']', 'error': 'Bad Request'}


FAILED: ValueError: Structured Output response does not have a 'parsed' field nor a 'refusal' field. Received message:

content='' additional_kwargs={'parsed': None, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 41, 'total_tokens': 165, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Sehyo/Qwen3.5-122B-A10B-NVFP4', 'system_fingerprint': None, 'id': 'chatcmpl-trace_bf7aaeea-b3bd-4af0-93a1-ca24516301d0', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e49d8-782c-77d1-8596-8991354749b1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 41, 'output_tokens': 124, 'total_tokens': 165, 'input_token_details': {}, 'output_token_details': {}}

--- json_mode ---


Traceback (most recent call last):
  File "/tmp/ipykernel_23035/4056871112.py", line 42, in <module>
    out = structured.invoke(prompt)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: Structured Output response does not have a 'parsed' field nor a 'refusal' field. Received message:

content='' additional_kwargs={'parsed': None, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 41, 'total_tokens': 165, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Sehyo/Qwen3.5-122B-A10B-NVFP4', 'system_fingerprint': None, 'id': 'chatcmpl-trace_bf7aaeea-b3bd-4af0-93a1-ca24516301d0', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e49d8-782c-77d1-8596-8991354749b1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 41, 'output_tokens': 124, 'total_tokens': 165, 'input_token_details': {}, 'output_token_details': {}}


FAILED: OutputParserException: Invalid json output: 
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

Structured output failed for all methods.
Baseline (no schema):


Traceback (most recent call last):
  File "/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/langchain_core/output_parsers/json.py", line 88, in parse_result
    return parse_json_markdown(text)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/tmp/ipykernel_23035/4056871112.py", line 42, in <module>
    out = structured.invoke(prompt)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
langchain_core.exceptions.OutputParserException: Invalid json output: 
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 




{"title": "Test", "description": "ok"}
